# TFG — Experimentos SELD DCASE2024 Task 3A

Notebook reorganizado para ejecutar experimentos de forma limpia, trazable y eficiente en Google Colab.

Objetivo:
- evitar recalcular features/labels innecesariamente;
- comprobar recursos antes de entrenar;
- ejecutar modelos con `job_id` distinto;
- registrar resultados comparables;
- mantener separadas las pruebas de arquitectura, loss y distancia.

## 0. Reglas rápidas de uso

### No ejecutar `batch_feature_extraction.py` salvo que cambies labels/features

No hace falta regenerar nada si cambias:
- arquitectura (`seldnet_model.py`);
- learning rate;
- dropout;
- número de epochs;
- loss;
- SE blocks;
- Frequency Attention.

Sí hace falta regenerar labels si cambias:
- `log`, `sqrt` o cualquier transformación de distancia en `cls_feature_class.py`;
- estructura de labels;
- formato ADPIT.

Sí hace falta regenerar features si cambias:
- mel bins;
- hop length;
- sampling rate;
- SALSA/GCC/features acústicas.

## 1. Comprobar GPU, RAM y entorno

In [ ]:
!nvidia-smi

Tue Jun 23 13:35:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   26C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import psutil
import torch
import gc
import os
from datetime import datetime

ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM disponible: {ram_gb:.1f} GB")
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

RAM disponible: 189.9 GB
CUDA disponible: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memoria GPU total (GB): 101.97


## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2.1 Facilitar lecturas archivos para train


In [ ]:
#1 SOLA EJECUCION-->LISTO

# 1. Crear directorio temporal en el disco rápido local de la máquina virtual
!mkdir -p /content/compresion_temporal/

# 2. Copiar la carpeta completa desde Drive al entorno local rápido
print("Copiando subcarpetas a local (operación masiva segura)...")
!cp -r /content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/seld_feat_label /content/compresion_temporal/

# 3. Entrar a la carpeta local y comprimir desde ahí dentro
%cd /content/compresion_temporal/
print("Comprimiendo dataset en local (gracias a la CPU del servidor esto volará)...")
!zip -r -q seld_feat_label.zip seld_feat_label

#4. Mover el archivo .zip final directamente a tu Google Drive
print("Guardando el archivo .zip definitivo en tu Google Drive...")
!cp seld_feat_label.zip /content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/

print("¡Proceso completado con éxito! Ya tienes 'seld_feat_label.zip' a salvo en tu Drive.")

Copiando subcarpetas a local (operación masiva segura)...
cp: cannot stat '/content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/seld_feat_label': No such file or directory
/content/compresion_temporal
Comprimiendo dataset en local (gracias a la CPU del servidor esto volará)...

zip error: Nothing to do! (try: zip -r -q seld_feat_label.zip . -i seld_feat_label)
Guardando el archivo .zip definitivo en tu Google Drive...
cp: cannot stat 'seld_feat_label.zip': No such file or directory
¡Proceso completado con éxito! Ya tienes 'seld_feat_label.zip' a salvo en tu Drive.


In [ ]:
import os

print("Iniciando volcado de datos al almacenamiento local rápido...")

# 1. Crear el directorio de destino local en la máquina virtual (fuera de Drive)
!mkdir -p /content/dataset_local/

# 2. Copiar el archivo zip desde Drive al disco rápido local
!cp "/content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/seld_feat_label.zip" /content/dataset_local/

# 3. Descomprimir el dataset en la carpeta local de forma silenciosa (-q)
print("Descomprimiendo archivos en local (esto será casi instantáneo)...")
!unzip -q /content/dataset_local/seld_feat_label.zip -d /content/dataset_local/

# 4. Verificación de seguridad
print("Contenido de /content/dataset_local/:")
!ls -l /content/dataset_local/

Iniciando volcado de datos al almacenamiento local rápido...
Descomprimiendo archivos en local (esto será casi instantáneo)...
Contenido de /content/dataset_local/:
total 8547736
drwx------ 5 root root       4096 May 17 09:58 seld_feat_label
-rw------- 1 root root 8752870747 Jun 23 13:38 seld_feat_label.zip


## 3. Entrar al repositorio del baseline

In [ ]:
%cd /content/drive/MyDrive/TFG/DCASE2024_seld_baseline
!pwd
!ls

/content/drive/MyDrive/TFG/DCASE2024_seld_baseline
/content/drive/MyDrive/TFG/DCASE2024_seld_baseline
3_1_dev_split0_multiaccdoa_foa_model.h5
6_1_dev_split0_multiaccdoa_mic_gcc_model.h5
archivos_a_corregir.txt
backup
batch_feature_extraction.py
cls_compute_seld_results.py
cls_data_generator.py
cls_feature_class.py
cls_vid_features.py
images
models_audio
parameters_backup_20260517_120118.py
parameters_backup_20260517_141442.py
parameters_backup_20260524_132000.py
parameters_backup_20260524_154819.py
parameters_backup_20260524_173032.py
parameters_backup_20260524_185058.py
parameters_backup_20260623_061539.py
parameters.py
__pycache__
README.md
results_audio
SELD_evaluation_metrics.py
seldnet_model_backup_20260517_120118.py
seldnet_model_backup_20260517_141442.py
seldnet_model_backup_20260524_132000.py
seldnet_model_backup_20260524_154819.py
seldnet_model_backup_20260524_173032.py
seldnet_model_backup_20260524_185058.py
seldnet_model_backup_20260623_061539.py
seldnet_model.py
train_seldn

## 4. Comprobar dataset y espacio disponible

In [ ]:
!ls /content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/

foa_dev  metadata_dev  seld_feat_label	seld_feat_label.zip


In [ ]:
!df -h /content/drive

Filesystem      Size  Used Avail Use% Mounted on
drive           236G   82G  155G  35% /content/drive


## 5. Parche NumPy 2.0

Ejecutar una vez por seguridad. Sustituye `np.NaN` por `np.nan` si aparece.

In [ ]:
!grep -R "np.NaN" -n . || echo "OK: no aparece np.NaN"
!grep -RIl "np.NaN" . | xargs -r sed -i 's/np.NaN/np.nan/g'

^C
^C


In [ ]:
!echo "Corrigiendo solo SELD_evaluation_metrics.py..."
!python - <<'PY'
from pathlib import Path

p = Path("SELD_evaluation_metrics.py")
text = p.read_text()

print("Apariciones antes:", text.count("np.NaN"))

text = text.replace("np.NaN", "np.nan")
p.write_text(text)

text2 = p.read_text()
print("Apariciones después:", text2.count("np.NaN"))
print("OK")


Corrigiendo solo SELD_evaluation_metrics.py...
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Apariciones antes: 0
Apariciones después: 0
OK


## 6. Verificar configuración actual

Ejecuta esta celda antes de cada experimento.

In [ ]:
!python -c "import parameters; p=parameters.get_params('32'); print('quick_test=', p['quick_test']); print('nb_epochs=', p['nb_epochs']); print('lr=', p['lr']); print('dropout=', p['dropout_rate']); print('dataset=', p['dataset']); print('multi_accdoa=', p['multi_accdoa']); print('foa_channel_mode=', p.get('foa_channel_mode')); print('feat_label_dir=', p['feat_label_dir'])"

## 7. Limpieza de memoria antes de entrenar

Ejecutar antes de lanzar un nuevo entrenamiento, especialmente si has entrenado varios modelos seguidos.

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memoria limpiada.")

Memoria limpiada.


# A. Extracción de features/labels

## Ejecutar SOLO si has cambiado features o labels

Para experimentos de arquitectura como SE blocks o Frequency Attention, NO ejecutes esta sección.

In [ ]:
# Ejecutar solo si has cambiado cls_feature_class.py o parámetros de features
# !python batch_feature_extraction.py 3

# B. Experimento baseline / entrenamiento genérico

Usa siempre un `job_id` distinto para no machacar modelos anteriores.

Ejemplos:
- `baseline_5ep`
- `se_10ep_lr1e3`
- `se_10ep_lr5e4`
- `se_freq_10ep`

In [ ]:
JOB_ID = "se_freq_10ep"  # cambiar este nombre antes de cada experimento
print("JOB_ID =", JOB_ID)

In [ ]:
!python train_seldnet.py 3 {JOB_ID}

# C. Tabla de resultados acumulados

Actualizada manualmente después de cada entrenamiento.

| Experimento | Cambio principal | Epochs | LR | Dropout | SELD | F-score | AE | Dist err | Rel dist | Conclusión |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|
| Baseline | Original | 5 | 1e-3 | 0.05 | 0.78 | 5.1 | 29.7 | 0.52 | 0.27 | Referencia |
| Log scaling | Labels distancia log | 5 | 1e-3 | 0.05 | 0.79 | 4.5 | 29.8 | 0.59 | 0.30 | No mejora |
| Weight 0.5 | Loss ponderada distancia | 5 | 1e-3 | 0.05 | 0.81 | 4.8 | 31.7 | 0.88 | 0.44 | Empeora distancia |
| Weight 2.0 | Loss ponderada distancia | 5 | 1e-3 | 0.05 | 0.85 | 3.1 | 28.0 | 0.99 | 0.44 | Empeora global |
| Sqrt scaling | Labels distancia sqrt | 5 | 1e-3 | 0.05 | 0.77 | 4.8 | 28.7 | 0.77 | 0.42 | Trade-off |
| SE blocks | Atención canal | 5 | 1e-3 | 0.05 | 0.76 | 4.1 | 29.6 | 0.75 | 0.38 | Mejora leve |
| SE blocks | Atención canal | 10 | 1e-3 | 0.05 | 0.66 | 7.4 | 31.8 | 0.57 | 0.29 | Mejor actual |
| SE blocks | Atención canal | 10 | 5e-4 | 0.05 | 0.75 | 5.4 | 28.7 | 0.71 | 0.36 | LR bajo no mejora |
| SE + FreqAtt | Canal + frecuencia | 10 | 1e-3 | 0.05 | 0.74 | 7.4 | 30.4° | 0.64 | 0.29 |  atención frecuencial aporta información útil para estabilizar la representación espectral |  
| SE + FreqAtt | Canal + frecuencia | 20 | 1e-3 | 0.05 | 0.65 | 9.2 | 32.6° | 0.53 | 0.29 |  |  




# D. Comprobaciones de código antes de entrenar

## 1. Ver si el modelo tiene SEBlock

In [ ]:
!grep -n "class SEBlock\|self.se\|SEBlock" seldnet_model.py

## 2. Ver si el modelo tiene FrequencyAttention

In [ ]:
!grep -n "class FrequencyAttention\|freq_att" seldnet_model.py || echo "FrequencyAttention no está activado"

## 3. Comprobar que la loss es la original o la experimental esperada

In [ ]:
!grep -n "criterion = seldnet_model" train_seldnet.py

## 4. Comprobar si quedan transformaciones de distancia activas

Para arquitectura SE/Frequency Attention, debería NO aparecer `log1p`, `sqrt`, `expm1` ni `** 2` en los sitios críticos.

In [ ]:
!grep -R "log1p\|expm1\|sqrt\|dist0 = dist0 \*\* 2\|dist1 = dist1 \*\* 2\|dist2 = dist2 \*\* 2" cls_feature_class.py train_seldnet.py || echo "OK: no hay transformaciones de distancia detectadas"

# E. Experimentación


In [ ]:
!python -c "import parameters; p=parameters.get_params('3'); print(p['nb_epochs'], p['dropout_rate'], p['lr'], p['T_max'], p['eta_min'])"
!grep -n "SEBlock\\|FrequencyAttention\\|freq_att" seldnet_model.py

SET: 3
FOA + multi ACCDOA

	quick_test: False
	finetune_mode: True
	pretrained_model_weights: 3_1_dev_split0_multiaccdoa_foa_model.h5
	dataset_dir: /content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/
	feat_label_dir: /content/dataset_local/seld_feat_label/
	model_dir: models_audio
	dcase_output_dir: results_audio
	mode: dev
	dataset: foa
	fs: 24000
	hop_len_s: 0.02
	label_hop_len_s: 0.1
	max_audio_len_s: 60
	nb_mel_bins: 64
	use_salsalite: False
	fmin_doa_salsalite: 50
	fmax_doa_salsalite: 2000
	fmax_spectra_salsalite: 9000
	modality: audio
	multi_accdoa: True
	thresh_unify: 15
	label_sequence_length: 50
	batch_size: 128
	dropout_rate: 0.05
	nb_cnn2d_filt: 64
	f_pool_size: [4, 4, 2]
	nb_heads: 8
	nb_self_attn_layers: 2
	nb_transformer_layers: 2
	nb_rnn_layers: 2
	rnn_size: 128
	nb_fnn_layers: 1
	fnn_size: 128
	nb_epochs: 30
	lr: 0.001
	foa_channel_mode: iv_only
	eta_min: 1e-05
	T_max: 30
	average: macro
	segment_based_metrics: False
	evaluate_distance: True
	lad_doa_thresh: 20
	lad_dist

In [ ]:
!grep -n "FrequencyAttention\|SEBlock\|CosineAnnealingLR" seldnet_model.py train_seldnet.py || echo "OK: baseline puro"

OK: baseline puro


In [ ]:
JOB_ID = "baseline_xyz_only_40ep"
!python train_seldnet.py 34 {JOB_ID}

['train_seldnet.py', '34', 'baseline_xyz_only_40ep']
SET: 34
BASELINE FOA ABLATION - XYZ ONLY

	quick_test: False
	finetune_mode: True
	pretrained_model_weights: 3_1_dev_split0_multiaccdoa_foa_model.h5
	dataset_dir: /content/drive/MyDrive/TFG/DCASE2024_SELD_dataset/
	feat_label_dir: /content/dataset_local/seld_feat_label/
	model_dir: models_audio
	dcase_output_dir: results_audio
	mode: dev
	dataset: foa
	fs: 24000
	hop_len_s: 0.02
	label_hop_len_s: 0.1
	max_audio_len_s: 60
	nb_mel_bins: 64
	use_salsalite: False
	fmin_doa_salsalite: 50
	fmax_doa_salsalite: 2000
	fmax_spectra_salsalite: 9000
	modality: audio
	multi_accdoa: True
	thresh_unify: 15
	label_sequence_length: 50
	batch_size: 128
	dropout_rate: 0.05
	nb_cnn2d_filt: 64
	f_pool_size: [4, 4, 2]
	nb_heads: 8
	nb_self_attn_layers: 2
	nb_transformer_layers: 2
	nb_rnn_layers: 2
	rnn_size: 128
	nb_fnn_layers: 1
	fnn_size: 128
	nb_epochs: 40
	lr: 0.001
	foa_channel_mode: xyz_only
	eta_min: 1e-05
	T_max: 40
	average: macro
	segment_based_

# F. Guardar versiones de código

Ejecuta esto cuando un modelo salga bien.

In [ ]:
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
!cp seldnet_model.py seldnet_model_backup_{stamp}.py
!cp parameters.py parameters_backup_{stamp}.py
print("Backups guardados con timestamp:", stamp)

Backups guardados con timestamp: 20260623_145426


# G. Apagar/liberar recursos al terminar

Cuando acabes una tanda de experimentos:
1. Guarda resultados.
2. Descarga/copias backups importantes.
3. Libera memoria.
4. Desconecta runtime si no vas a usarlo.

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos de Python/GPU liberados parcialmente. Para liberar todo: Entorno de ejecución > Desconectar y eliminar entorno de ejecución.")

Recursos de Python/GPU liberados parcialmente. Para liberar todo: Entorno de ejecución > Desconectar y eliminar entorno de ejecución.
